In [0]:
from pyspark.sql.functions import col, explode, sum as spark_sum, lit

# Get current widget values
selected_date = dbutils.widgets.get("Select_Date")

# Map date to folder
store_file_map = {
    "2025-09-06": "/Volumes/workspace/default/covid1/poc_files/raw/1",
    "2025-09-07": "/Volumes/workspace/default/covid1/poc_files/raw/2",
    "2025-09-08": "/Volumes/workspace/default/covid1/poc_files/raw/3"
}

selected_file_path = store_file_map[selected_date]
print("Reading files from folder:", selected_file_path)

# Re-read the JSON files based on current widget value
df2 = spark.read.option("multiLine", True).json(selected_file_path + "/*.json")

# Explode card payments
df_payment_card = df2.select(explode(col("payments.cards")).alias("card"))

# Explode cash payments
df_payment_cash = df2.select(explode(col("payments.cash")).alias("cash_amount"))

# Summarize card totals
df_card_sum = (
    df_payment_card
    .filter(col("card.totalAmount").cast("double") > 0)
    .groupby(col("card.cardType").alias("payment_type"))
    .agg(spark_sum(col("card.totalAmount").cast("double")).alias("totalAmount"))
)

# Summarize cash totals
df_cash_sum = (
    df_payment_cash
    .filter(col("cash_amount").cast("double") > 0)
    .agg(spark_sum(col("cash_amount").cast("double")).alias("totalAmount"))
    .withColumn("payment_type", lit("cash"))
)

# Combine card + cash
df_ans = df_card_sum.unionByName(df_cash_sum)

# Add deployment name and date
df_final = (
    df_ans
    .withColumn("deployment_name", lit(deployment_name))
    .withColumn("date", lit(selected_date))
    .select("deployment_name", "date", *df_ans.columns)
)

# Total amount sum
total_amount_sum = df_final.select(spark_sum(col("totalAmount").cast("double")).alias("total_amount_sum")).collect()
print(total_amount_sum)
display(total_amount_sum)

In [0]:
# import os

# selected_file_path = "/Volumes/workspace/default/covid1/poc_files/raw/"

# files = [os.path.join(selected_file_path, f) for f in os.listdir(selected_file_path) if f.endswith(".json")]
# print("Files being read:")
# for f in files:
#     print(f)
